# TEST-06 POSTFIX: Drawdown Semantics

WP-5b confirms that previously unbounded drawdown semantics are now bounded by liquidation.

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
from lib.backtest import run_backtest
from lib.evaluation import max_drawdown

pre_curve = pd.Series(
    [10_000.0, 11_000.0, 12_000.0, 5_000.0, -10_000.0, -40_000.0],
    index=pd.date_range('2020-01-01', periods=6, freq='4h', tz='UTC'),
)
pre_dd, _, _ = max_drawdown(pre_curve)
print('=== WP-5a negative-equity drawdown formula ===')
print(f'Pre-fix semantic curve: {pre_curve.tolist()}')
print(f'Pre-fix max DD:         {pre_dd:.6f} ({pre_dd*100:.2f}%)')
print('This exceeds 100% because equity was allowed below zero.')


=== WP-5a negative-equity drawdown formula ===
Pre-fix semantic curve: [10000.0, 11000.0, 12000.0, 5000.0, -10000.0, -40000.0]
Pre-fix max DD:         4.333333 (433.33%)
This exceeds 100% because equity was allowed below zero.


In [ ]:
def synthetic_ohlcv(closes):
    idx = pd.date_range('2020-01-01', periods=len(closes), freq='1h', tz='UTC')
    opens = [closes[0]] + list(closes[:-1])
    return pd.DataFrame({
        'open': opens,
        'high': [max(o, c) for o, c in zip(opens, closes)],
        'low': [min(o, c) for o, c in zip(opens, closes)],
        'close': closes,
        'volume': [1000.0] * len(closes),
    }, index=idx)

ohlcv = synthetic_ohlcv([100.0, 100.0, 20_000.0, 30_000.0, 40_000.0])
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))
signal = pd.Series([0.0, -1.0, -1.0, -1.0, -1.0], index=ohlcv.index)
post = run_backtest(signal, ohlcv, funding, fee_bps_taker=0.0, fee_bps_maker=0.0,
                    slippage_bps=0.0, initial_equity=10_000.0,
                    stream='klines', interval='1h')
post_dd, _, _ = max_drawdown(post.equity_curve)
print('=== WP-5b bounded-equity rerun ===')
print(f'Post-fix equity curve:       {[float(v) for v in post.equity_curve.tolist()]}')
print(f'Post-fix max DD:             {post_dd:.6f} ({post_dd*100:.2f}%)')
print(f'Liquidated:                  {post.liquidated}')
print(f'Liquidation timestamp:       {post.liquidation_timestamp}')
print(f'Final equity:                {float(post.equity_curve.iloc[-1]):.2f}')
print(f'Equity minimum:              {float(post.equity_curve.min()):.2f}')
print(f'Trades after liquidation:    {int((post.trades_df["entry_bar"] > post.liquidation_timestamp).sum()) if not post.trades_df.empty else 0}')


=== WP-5b bounded-equity rerun ===
Post-fix equity curve:       [10000.0, 10000.0, 0.0, 0.0, 0.0]
Post-fix max DD:             1.000000 (100.00%)
Liquidated:                  True
Liquidation timestamp:       2020-01-01 02:00:00+00:00
Final equity:                0.00
Equity minimum:              0.00
Trades after liquidation:    0


In [ ]:
verdict = 'PASS' if pre_dd > 1.0 and post_dd <= 1.0 and post.liquidated and post.equity_curve.min() >= 0.0 else 'FAIL'
print(f'=== VERDICT: {verdict} ===')
print('DD is bounded to <= 100%.')
print('Liquidation event is recorded.')
print('Equity is floored at 0.0 for the remainder of the window.')


=== VERDICT: PASS ===
DD is bounded to <= 100%.
Liquidation event is recorded.
Equity is floored at 0.0 for the remainder of the window.
